# 05b — Ablation Study: LSTM-only vs GAT-only vs No-Fusion vs Full-Fused

**Week 5 | RAKSHAK-ICS**

This notebook benchmarks **4 configurations** of the Blue Agent to understand each stream's contribution:

| Config | Description |
|---|---|
| **LSTM-only** | α=1.0 — only LSTM-AE stream (Stream 1) |
| **GAT-only** | α=0.0 — only GAT GNN stream (Stream 2) |
| **No fusion (max)** | max(lstm_norm, gnn_norm), τ=0.5 |
| **Full fused** | α=optimal, τ=optimal (tuned on val) |

All configs evaluated over **5 seeds** with mean±std reporting.

## 1. Setup & Imports

In [ ]:
import sys, json, warnings, logging, time
from pathlib import Path
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

ROOT = Path("..") if Path("../src").exists() else Path(".")
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.style as mstyle
mstyle.use("dark_background")
import torch

from src.gnn import SensorGAT, train_gat, compute_gat_scores, tune_gat_threshold
from src.lstm_ae import LSTMAutoencoder, train_lstm_ae, compute_reconstruction_scores, LSTMBlueAgent
from src.fusion import tune_fusion, run_ablation, save_ablation_results, save_fusion_params
from src.stat_utils import set_all_seeds, format_result

PROOF   = ROOT / "data" / "proof"
MODELS  = ROOT / "models"
FIGURES = ROOT / "results" / "figures"
TABLES  = ROOT / "results" / "tables"

SEEDS = [42, 123, 456, 789, 1024]
ANOMALY_RATIO = 0.05
DEVICE = torch.device("cpu")

print("Setup complete ✓")

## 2. Load Data

In [ ]:
X_train_nf  = np.load(PROOF / "node_features_train.npy")
X_val_nf    = np.load(PROOF / "node_features_val.npy")
X_test_nf   = np.load(PROOF / "node_features_test.npy")
edge_index  = np.load(PROOF / "edge_index.npy")
edge_weights = np.load(PROOF / "edge_weights.npy")

X_train = np.load(PROOF / "X_train.npy")
X_val   = np.load(PROOF / "X_val.npy")
X_test  = np.load(PROOF / "X_test.npy")

def inject_anomalies(X, ratio=0.05, seed=42):
    rng = np.random.default_rng(seed)
    X_a = X.copy(); n = len(X_a)
    idx = rng.choice(n, size=int(n*ratio), replace=False)
    for i in idx:
        f = rng.integers(0, X_a.shape[1])
        X_a[i, f] += rng.normal(0, 3.0 * X_a[:, f].std())
    y = np.zeros(n, dtype=int); y[idx] = 1
    return X_a, y

_, y_val  = inject_anomalies(X_val_nf,  ratio=ANOMALY_RATIO, seed=99)
_, y_test = inject_anomalies(X_test_nf, ratio=ANOMALY_RATIO, seed=42)
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Graph: {edge_index.shape[1]} edges")

## 3. Multi-Seed Ablation

In [ ]:
ablation_per_seed = {cfg: [] for cfg in ["lstm_only", "gat_only", "no_fusion_max", "full_fused"]}
best_alpha_global, best_tau_global = 0.5, 0.5

for seed in SEEDS:
    print(f"\n{'='*60}  SEED {seed}  {'='*60}")
    set_all_seeds(seed)

    # ── Train LSTM-AE ──────────────────────────────────────────
    print("  [1/2] Training LSTM-AE...")
    lstm_model = LSTMAutoencoder(input_dim=65)
    lstm_history = train_lstm_ae(
        lstm_model, X_train, X_val,
        epochs=30, patience=10, batch_size=256, subsample=3, device=DEVICE,
    )
    lstm_val_sc  = compute_reconstruction_scores(lstm_model, X_val,  device=DEVICE)
    # Augmented val/test for score computation
    X_val_aug, _  = inject_anomalies(X_val,  ANOMALY_RATIO, seed+1)
    X_test_aug, _ = inject_anomalies(X_test, ANOMALY_RATIO, seed)
    lstm_val_sc_aug  = compute_reconstruction_scores(lstm_model, X_val_aug,  device=DEVICE)
    lstm_test_sc_aug = compute_reconstruction_scores(lstm_model, X_test_aug, device=DEVICE)
    print(f"  LSTM-AE val score: {lstm_val_sc.mean():.6f}")

    # ── Train GAT ──────────────────────────────────────────────
    print("  [2/2] Training GAT GNN...")
    gat_model = SensorGAT(node_feature_dim=5, hidden_dim=16, num_heads=8, dropout=0.1, edge_dim=1)
    gat_history = train_gat(
        gat_model, X_train_nf, edge_index, edge_weights, X_val_nf,
        epochs=30, patience=15, subsample=3, device=DEVICE,
    )
    gat_val_sc   = compute_gat_scores(gat_model, X_val_nf,  edge_index, edge_weights, device=DEVICE)
    X_val_nf_aug, _  = inject_anomalies(X_val_nf,  ANOMALY_RATIO, seed+1)
    X_test_nf_aug, _ = inject_anomalies(X_test_nf, ANOMALY_RATIO, seed)
    gat_val_sc_aug   = compute_gat_scores(gat_model, X_val_nf_aug,  edge_index, edge_weights, device=DEVICE)
    gat_test_sc_aug  = compute_gat_scores(gat_model, X_test_nf_aug, edge_index, edge_weights, device=DEVICE)
    print(f"  GAT val score: {gat_val_sc.mean():.6f}")

    # ── Tune fusion on val ─────────────────────────────────────
    best_alpha, best_tau, best_f1, _ = tune_fusion(
        lstm_val_sc_aug, gat_val_sc_aug, y_val,
        alpha_step=0.1, threshold_step=0.05,
    )
    if seed == 42:
        best_alpha_global, best_tau_global = best_alpha, best_tau
    print(f"  Best α={best_alpha:.2f}, τ={best_tau:.3f}, F1={best_f1:.4f}")

    # ── Run 4-config ablation ──────────────────────────────────
    seed_ablation = run_ablation(
        lstm_test_sc_aug, gat_test_sc_aug,
        lstm_val_sc_aug, gat_val_sc_aug,
        y_test, y_val, best_alpha, best_tau,
    )
    for cfg, metrics in seed_ablation.items():
        ablation_per_seed[cfg].append({"seed": seed, **metrics})

print("\n✓ All seeds complete")

## 4. Aggregate Results

In [ ]:
metric_keys = ["f1", "precision", "recall", "auc_roc"]
aggregated = {}
for cfg, per_seed_list in ablation_per_seed.items():
    agg = {}
    for k in metric_keys:
        vals = [r[k] for r in per_seed_list]
        m, s = float(np.mean(vals)), float(np.std(vals))
        agg[k] = {"scores": vals, "mean": m, "std": s, "formatted": f"{m:.3f}\u00b1{s:.3f}"}
    aggregated[cfg] = agg

config_labels = {
    "lstm_only": "LSTM-only (α=1.0)",
    "gat_only":  "GAT-only (α=0.0)",
    "no_fusion_max": "No-Fusion (max)",
    "full_fused": "Full Fused (α=opt)",
}

print("="*70)
print(f"{'Ablation Study — 5-Seed Results':^70}")
print("="*70)
print(f"{'Config':<25} {'F1':>12} {'Precision':>12} {'Recall':>12} {'AUC-ROC':>12}")
print("-"*70)
for cfg, agg in aggregated.items():
    label = config_labels[cfg]
    print(f"{label:<25} {agg['f1']['formatted']:>12} {agg['precision']['formatted']:>12} "
          f"{agg['recall']['formatted']:>12} {agg['auc_roc']['formatted']:>12}")
print("="*70)

# Save
save_ablation_results(aggregated, path=str(TABLES / "ablation_results.json"))
save_fusion_params(best_alpha_global, best_tau_global, {}, path=str(MODELS / "fusion_params.json"))
print(f"\nSaved ablation_results.json and fusion_params.json")

## 5. Ablation Bar Chart

In [ ]:
cfg_names = list(aggregated.keys())
cfg_labels_list = [config_labels[c] for c in cfg_names]
f1_means = [aggregated[c]["f1"]["mean"] for c in cfg_names]
f1_stds  = [aggregated[c]["f1"]["std"]  for c in cfg_names]
colors   = ['#4FC3F7', '#A5D6A7', '#FF8A65', '#CE93D8']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# F1 bar chart
bars = axes[0].bar(cfg_labels_list, f1_means, yerr=f1_stds,
                   color=colors, edgecolor='white', capsize=6, linewidth=0.5)
axes[0].set_ylabel('F1 Score', color='white')
axes[0].set_title('Ablation — F1 per Config (mean±std)', color='white', fontsize=12)
axes[0].tick_params(axis='x', rotation=15, colors='white')
axes[0].tick_params(axis='y', colors='white')
axes[0].set_facecolor('#1E1E1E')
for bar, val, err in zip(bars, f1_means, f1_stds):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + err + 0.001,
                 f'{val:.3f}', ha='center', va='bottom', color='white', fontsize=9)

# Multi-metric grouped bar chart
x = np.arange(len(cfg_names))
width = 0.2
for i, (mk, mc) in enumerate(zip(metric_keys, colors)):
    means = [aggregated[c][mk]["mean"] for c in cfg_names]
    stds  = [aggregated[c][mk]["std"]  for c in cfg_names]
    axes[1].bar(x + i*width, means, width, label=mk, color=mc, edgecolor='white',
                yerr=stds, capsize=3, linewidth=0.5, alpha=0.85)
axes[1].set_xticks(x + width*1.5)
axes[1].set_xticklabels(cfg_labels_list, rotation=12, ha='right', color='white')
axes[1].tick_params(axis='y', colors='white')
axes[1].set_ylabel('Score', color='white')
axes[1].set_title('Ablation — All Metrics', color='white', fontsize=12)
axes[1].legend(facecolor='#1E1E1E', labelcolor='white')
axes[1].set_facecolor('#1E1E1E')

for ax in axes:
    for spine in ax.spines.values(): spine.set_color('#555555')
fig.patch.set_facecolor('#121212')
plt.tight_layout()
out_path = FIGURES / "ablation_results.png"
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#121212')
plt.show()
print(f"Saved → {out_path}")

## 6. Summary

### Ablation Findings
The 4-config ablation demonstrates each component's contribution:

| Config | What it measures |
|---|---|
| **LSTM-only** | Temporal reconstruction quality alone |
| **GAT-only** | Spatial inter-sensor correlation alone |
| **No fusion (max)** | Both streams without learned combination |
| **Full fused** | Optimal combination via α-sweep |

The **Full Fused** configuration should outperform either stream alone,
validating the dual-stream architecture.

### Files Saved
- `results/tables/ablation_results.json` — 5-seed aggregated metrics
- `models/fusion_params.json` — best α and τ (seed 42)
- `results/figures/ablation_results.png` — bar charts

**Next**: `06_fusion_eval.ipynb` — full α-sweep heatmap + HAI cross-domain evaluation
